### Round 3 Analysis — VELVETFRUIT Options (VEV_*)

In [ ]:
import os
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import scipy.stats as stats
from scipy.stats import norm
from scipy.optimize import brentq, curve_fit

from pathlib import Path
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.stattools import jarque_bera

import prosperity4
from prosperity4.utils.statistics_utils import compute_returns
from prosperity4.utils.dataloader import (
    load_trading_data,
    get_product_data,
    get_day_data,
    get_product_day_data,
    get_price_data,
    get_order_book_data,
    get_volume_data,
    convert_timestamp,
)

plt.style.use("dark_background")
sns.set_palette("pastel")

### Data Loading

In [ ]:
REPO_ROOT = Path(prosperity4.__file__).parents[1]
DATA_FOLDER = REPO_ROOT / "prosperity4" / "round3" / "data"
ROUND_NUM = 3
DAYS = [0, 1, 2]

data = load_trading_data(DATA_FOLDER, ROUND_NUM, DAYS)
prices_df = data.get("prices")
trades_df = data.get("trades")

print("Prices Shape :", prices_df.shape if prices_df is not None else None)
print("Trades Shape :", trades_df.shape if trades_df is not None else None)
print("\n--- Prices Head ---")
display(prices_df.head())
print("\n--- Trades Head ---")
display(trades_df.head())

### Options Data — Extract & Prepare

In [ ]:
OPTION_STRIKES = [4000, 4500, 5000, 5100, 5200, 5300, 5400, 5500, 6000, 6500]
OPTION_NAMES   = [f"VEV_{k}" for k in OPTION_STRIKES]
UNDERLYING     = "VELVETFRUIT_EXTRACT"
T_DAYS         = 5      # total days to expiry at t=0
R              = 0.0   # risk-free rate

options_df    = prices_df[prices_df["product"].isin(OPTION_NAMES)].copy()
underlying_df = prices_df[prices_df["product"] == UNDERLYING].copy()

# Parse strike from product name
options_df["strike"] = options_df["product"].str.extract(r"(\d+)$").astype(int)

# Time to expiry in years; within each day timestamps run 0–999900 (step=100)
# so 1 competition day = 1_000_000 timestamp units
options_df["T"] = (
    (T_DAYS - options_df["day"] - options_df["timestamp"] / 1_000_000) / 365.0
).clip(lower=1e-6)

print(f"Option products found : {sorted(options_df['product'].unique())}")
print(f"Options rows          : {len(options_df):,}")
print(f"Underlying rows       : {len(underlying_df):,}")

In [ ]:
# Merge underlying spot price into options on (day, timestamp)
spot = underlying_df[["day", "timestamp", "mid_price"]].rename(columns={"mid_price": "S"})
options_df = options_df.merge(spot, on=["day", "timestamp"], how="left")

# Market option price = mid_price, replace 0 with NaN
options_df["C"] = options_df["mid_price"].replace(0, np.nan)
options_df["S"] = options_df["S"].replace(0, np.nan)

# Convert to continuous timestamp axis
options_df = convert_timestamp(options_df)

display(
    options_df[["product", "strike", "day", "T", "S", "C",
                "bid_price_1", "ask_price_1"]].head(20)
)

### Option Price Analysis

In [ ]:
colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(OPTION_STRIKES)))

fig, axes = plt.subplots(5, 2, figsize=(16, 22))
axes = axes.flatten()

for i, (strike, name) in enumerate(zip(OPTION_STRIKES, OPTION_NAMES)):
    opt = options_df[options_df["product"] == name]
    axes[i].plot(opt["t"], opt["C"], color=colors[i], linewidth=0.7)
    axes[i].fill_between(
        opt["t"],
        opt["bid_price_1"].replace(0, np.nan),
        opt["ask_price_1"].replace(0, np.nan),
        alpha=0.2, color=colors[i], label="Bid-Ask spread"
    )
    axes[i].set_title(f"{name}  (K = {strike})", color="white")
    axes[i].set_ylabel("Price")
    axes[i].grid(True, alpha=0.3)
    axes[i].legend(fontsize=7)

plt.suptitle("Option Mid Prices Over Time (all strikes)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# All options on a single chart, coloured by strike
fig, ax = plt.subplots(figsize=(15, 6))
for i, (strike, name) in enumerate(zip(OPTION_STRIKES, OPTION_NAMES)):
    opt = options_df[options_df["product"] == name]
    ax.plot(opt["t"], opt["C"], color=colors[i], linewidth=0.8,
            label=f"K={strike}", alpha=0.85)

ax.set_title("All Option Mid Prices — Coloured by Strike", fontsize=13)
ax.set_xlabel("Time")
ax.set_ylabel("Option Price")
ax.legend(loc="upper right", fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Implied Volatility

In [ ]:
def bs_call(S, K, T, r, sigma):
    if T <= 1e-8 or sigma <= 1e-8:
        return max(S - K * np.exp(-r * T), 0.0)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)


def implied_vol(C_market, S, K, T, r=0.0):
    if any(np.isnan(v) for v in [C_market, S]) or S <= 0 or T <= 0:
        return np.nan
    intrinsic = max(S - K * np.exp(-r * T), 0.0)
    if C_market <= intrinsic or C_market >= S:
        return np.nan
    try:
        iv = brentq(
            lambda sigma: bs_call(S, K, T, r, sigma) - C_market,
            1e-6, 20.0, xtol=1e-7, maxiter=300
        )
        return iv if 0.0 < iv < 20.0 else np.nan
    except (ValueError, RuntimeError):
        return np.nan

In [ ]:
# Compute IV row-by-row
options_df["iv"] = options_df.apply(
    lambda row: implied_vol(row["C"], row["S"], row["strike"], row["T"], R),
    axis=1
)

iv_summary = (
    options_df.groupby(["product", "strike"])["iv"]
    .agg(iv_mean="mean", iv_median="median", iv_std="std", valid_rows="count")
    .round(4)
)
display(iv_summary)

In [ ]:
# IV over time — individual subplots
fig, axes = plt.subplots(5, 2, figsize=(16, 22))
axes = axes.flatten()

for i, (strike, name) in enumerate(zip(OPTION_STRIKES, OPTION_NAMES)):
    opt = options_df[options_df["product"] == name].dropna(subset=["iv"])
    axes[i].plot(opt["t"], opt["iv"], color=colors[i], linewidth=0.7)
    median_iv = opt["iv"].median()
    axes[i].axhline(
        median_iv, color="white", linewidth=1, linestyle="--",
        alpha=0.6, label=f"Median IV = {median_iv:.3f}"
    )
    axes[i].set_title(f"{name}  (K = {strike}) — Implied Volatility", color="white")
    axes[i].set_ylabel("IV")
    axes[i].grid(True, alpha=0.3)
    axes[i].legend(fontsize=7)

plt.suptitle("Implied Volatility Over Time (per Strike)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# IV over time — all strikes on one chart
fig, ax = plt.subplots(figsize=(15, 6))
for i, (strike, name) in enumerate(zip(OPTION_STRIKES, OPTION_NAMES)):
    opt = options_df[options_df["product"] == name].dropna(subset=["iv"])
    ax.plot(opt["t"], opt["iv"], color=colors[i], linewidth=0.5,
            alpha=0.8, label=f"K={strike}")

ax.set_title("Implied Volatility Over Time — All Strikes", fontsize=13)
ax.set_xlabel("Time")
ax.set_ylabel("Implied Volatility")
ax.legend(loc="upper right", fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Volatility Smile

In [ ]:
S_mean = options_df["S"].median()
print(f"Median VELVETFRUIT_EXTRACT spot price: {S_mean:.2f}")

smile_all = (
    options_df.groupby("strike")["iv"]
    .agg(iv_mean="mean", iv_median="median", iv_std="std")
    .reset_index()
)
smile_all["moneyness"] = np.log(smile_all["strike"] / S_mean)

smile_by_day = (
    options_df.groupby(["day", "strike"])["iv"]
    .median()
    .reset_index()
    .rename(columns={"iv": "iv_median"})
)
smile_by_day["moneyness"] = np.log(smile_by_day["strike"] / S_mean)

display(smile_all.round(4))

In [ ]:
day_colors = ["cyan", "orange", "lime"]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: smile by strike
ax = axes[0]
ax.plot(smile_all["strike"], smile_all["iv_median"],
        "o-", color="white", linewidth=2, markersize=7, label="All days (median IV)")
ax.fill_between(
    smile_all["strike"],
    smile_all["iv_median"] - smile_all["iv_std"],
    smile_all["iv_median"] + smile_all["iv_std"],
    alpha=0.15, color="white", label="\u00b11 std"
)
for d in DAYS:
    sub = smile_by_day[smile_by_day["day"] == d]
    ax.plot(sub["strike"], sub["iv_median"],
            "s--", color=day_colors[d], alpha=0.8, markersize=5, label=f"Day {d}")
ax.set_xlabel("Strike  K")
ax.set_ylabel("Implied Volatility")
ax.set_title("Volatility Smile — by Strike")
ax.legend()
ax.grid(True, alpha=0.3)

# Right: smile by log-moneyness
ax = axes[1]
ax.plot(smile_all["moneyness"], smile_all["iv_median"],
        "o-", color="white", linewidth=2, markersize=7, label="All days (median IV)")
ax.fill_between(
    smile_all["moneyness"],
    smile_all["iv_median"] - smile_all["iv_std"],
    smile_all["iv_median"] + smile_all["iv_std"],
    alpha=0.15, color="white", label="\u00b11 std"
)
for d in DAYS:
    sub = smile_by_day[smile_by_day["day"] == d]
    ax.plot(sub["moneyness"], sub["iv_median"],
            "s--", color=day_colors[d], alpha=0.8, markersize=5, label=f"Day {d}")
ax.axvline(0, color="gray", linestyle=":", alpha=0.6, label="ATM")
ax.set_xlabel("Log-Moneyness  log(K / S)")
ax.set_ylabel("Implied Volatility")
ax.set_title("Volatility Smile — by Log-Moneyness")
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle(
    f"Volatility Smile  (S \u2248 {S_mean:.0f},  T = {T_DAYS} days,  r = {R})",
    fontsize=13
)
plt.tight_layout()
plt.show()

### Smile Curve Fitting & Deviations

In [ ]:
# Fit a quadratic in log-moneyness: IV(x) = a + b*x + c*x^2
smile_fit = smile_all.dropna(subset=["iv_median"]).copy()
x_data = smile_fit["moneyness"].values
y_data = smile_fit["iv_median"].values

def smile_quad(x, a, b, c):
    return a + b * x + c * x**2

popt, pcov = curve_fit(smile_quad, x_data, y_data, p0=[0.3, 0.0, 0.5], maxfev=10000)
a, b, c = popt
perr = np.sqrt(np.diag(pcov))

x_fine = np.linspace(x_data.min() - 0.05, x_data.max() + 0.05, 400)
y_fine = smile_quad(x_fine, *popt)

y_pred    = smile_quad(x_data, *popt)
residuals = y_data - y_pred
rmse      = np.sqrt(np.mean(residuals**2))

print("Fitted quadratic smile:")
print(f"  IV(x) = {a:.4f}  +  {b:.4f}\u00b7x  +  {c:.4f}\u00b7x\u00b2")
print(f"  ATM vol (x=0) \u2248 {a:.4f}")
print(f"  Skew  (b)     = {b:.4f}  (\u00b1 {perr[1]:.4f})")
print(f"  Convexity (c) = {c:.4f}  (\u00b1 {perr[2]:.4f})")
print(f"  RMSE          = {rmse:.6f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(12, 10),
    gridspec_kw={"height_ratios": [2, 1]}
)

# Top: observed smile + fitted curve
ax1.scatter(x_data, y_data, color="cyan", s=90, zorder=5,
            label="Observed IV (median)")
ax1.plot(x_fine, y_fine, color="orange", linewidth=2,
         label=f"Fit: {a:.3f} + {b:.3f}\u00b7x + {c:.3f}\u00b7x\u00b2")
ax1.fill_between(x_fine, y_fine - rmse, y_fine + rmse,
                 color="orange", alpha=0.12, label=f"\u00b1RMSE ({rmse:.4f})")
ax1.axvline(0, color="gray", linestyle=":", alpha=0.5, label="ATM")
for xi, yi, strike in zip(x_data, y_data, smile_fit["strike"]):
    ax1.annotate(f"K={strike}", (xi, yi),
                 textcoords="offset points", xytext=(6, 4),
                 fontsize=7, color="white")
ax1.set_ylabel("Implied Volatility")
ax1.set_title("Volatility Smile — Observed vs Fitted Quadratic")
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# Bottom: residuals
bar_w = (x_data.max() - x_data.min()) / (len(x_data) + 1)
bar_colors = ["lime" if r >= 0 else "red" for r in residuals]
ax2.bar(x_data, residuals, color=bar_colors, alpha=0.85, width=bar_w)
ax2.axhline(0, color="white", linewidth=1, linestyle="--")
ax2.axhline( rmse, color="orange", linewidth=0.8, linestyle=":", alpha=0.7)
ax2.axhline(-rmse, color="orange", linewidth=0.8, linestyle=":", alpha=0.7)
for xi, ri, strike in zip(x_data, residuals, smile_fit["strike"]):
    offset = 5 if ri >= 0 else -12
    ax2.annotate(f"{ri:+.4f}", (xi, ri),
                 textcoords="offset points", xytext=(0, offset),
                 fontsize=7, color="white", ha="center")
ax2.set_xlabel("Log-Moneyness  log(K / S)")
ax2.set_ylabel("IV Residual")
ax2.set_title(f"Deviations from Fitted Smile  (RMSE = {rmse:.4f})")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Per-day smile fits — track how skew & convexity evolve toward expiry
fig, axes = plt.subplots(1, len(DAYS), figsize=(18, 5), sharey=True)

day_params = {}
for d, ax in zip(DAYS, axes):
    sub = smile_by_day[smile_by_day["day"] == d].dropna(subset=["iv_median"])
    if len(sub) < 4:
        ax.set_title(f"Day {d} — insufficient data")
        continue

    xd = sub["moneyness"].values
    yd = sub["iv_median"].values

    try:
        popt_d, _ = curve_fit(smile_quad, xd, yd, p0=[0.3, 0.0, 0.5], maxfev=5000)
        day_params[d] = popt_d
        x_fine_d = np.linspace(xd.min() - 0.05, xd.max() + 0.05, 300)
        ax.scatter(xd, yd, color=day_colors[d], s=70, zorder=5)
        ax.plot(x_fine_d, smile_quad(x_fine_d, *popt_d), color="white", linewidth=1.5)
        yd_pred = smile_quad(xd, *popt_d)
        rmse_d = np.sqrt(np.mean((yd - yd_pred)**2))
        ax.set_title(
            f"Day {d}  (T \u2248 {T_DAYS - d:.0f}d)\n"
            f"a={popt_d[0]:.3f}  b={popt_d[1]:.3f}  c={popt_d[2]:.3f}\n"
            f"RMSE={rmse_d:.4f}",
            fontsize=9
        )
    except RuntimeError:
        ax.set_title(f"Day {d} — fit failed")

    ax.axvline(0, color="gray", linestyle=":", alpha=0.5)
    ax.set_xlabel("Log-Moneyness")
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel("Implied Volatility")
plt.suptitle("Per-Day Volatility Smiles — Quadratic Fit", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Full summary table
summary = (
    options_df.groupby(["product", "strike"]).agg(
        iv_mean    = ("iv", "mean"),
        iv_median  = ("iv", "median"),
        iv_std     = ("iv", "std"),
        C_mean     = ("C",  "mean"),
        S_mean     = ("S",  "mean"),
        valid_rows = ("iv", "count"),
    )
    .round(4)
)
display(summary)